<a href="https://colab.research.google.com/github/charang9/SNOW-FLAKE-PROJECTS/blob/main/ML_P11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# PROJECT 11 — REAL-TIME NETWORK SECURITY LOGISTIC REGRESSION ENGINE
# ==========================================================

import requests
import io
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score

FLOW_LOGS_API_ENDPOINT = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
HOST_TELEMETRY_API_ENDPOINT = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"

# ==========================================================
# TASK 1 — API FETCH FUNCTION
# ==========================================================

def fetch_api_data(url: str) -> pd.DataFrame:

    response = requests.get(url)
    response.raise_for_status()

    df = pd.read_csv(io.StringIO(response.text))
    return df

# Fetch datasets
penguins = fetch_api_data(FLOW_LOGS_API_ENDPOINT)
iris = fetch_api_data(HOST_TELEMETRY_API_ENDPOINT)

# Clean data
penguins = penguins.dropna()

# Binary mapping
penguins["Is_Anomalous"] = penguins["species"].apply(
    lambda x: 0 if x == "Adelie" else 1
)

# Select features
df = penguins[
    [
        "bill_length_mm",
        "bill_depth_mm",
        "flipper_length_mm",
        "body_mass_g",
        "Is_Anomalous",
    ]
]

print(df.head())
print(df['Is_Anomalous'].value_counts())

print('\n')

print("[TASK 1 SUCCESS] API Data Fetched Successfully!\n")
print(f"Processed Network Logs Shape: {df.shape}\n")
print("Class Distribution (0: Normal / 1: Anomalous):")
print(df["Is_Anomalous"].value_counts())
print("\nFirst 3 Rows:")
print(df.head(3))





   bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g  Is_Anomalous
0            39.1           18.7              181.0       3750.0             0
1            39.5           17.4              186.0       3800.0             0
2            40.3           18.0              195.0       3250.0             0
4            36.7           19.3              193.0       3450.0             0
5            39.3           20.6              190.0       3650.0             0
Is_Anomalous
1    187
0    146
Name: count, dtype: int64


[TASK 1 SUCCESS] API Data Fetched Successfully!

Processed Network Logs Shape: (333, 5)

Class Distribution (0: Normal / 1: Anomalous):
Is_Anomalous
1    187
0    146
Name: count, dtype: int64

First 3 Rows:
   bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g  Is_Anomalous
0            39.1           18.7              181.0       3750.0             0
1            39.5           17.4              186.0       3800.0             0
2            40.3     

In [ ]:
# ==========================================================
# TASK 2 — TRAIN TEST SPLIT + SCALING
# ==========================================================

X = df.drop("Is_Anomalous", axis=1)
y = df["Is_Anomalous"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n[TASK 2 SUCCESS] Data Partitioned & Scaled.\n")
print(f"Training Features Shape : {X_train_scaled.shape}")
print(f"Testing Features Shape  : {X_test_scaled.shape}")

train_counts = y_train.value_counts()
test_counts = y_test.value_counts()

print(
    f"Train Target Balance    : 0 -> {train_counts[0]} | 1 -> {train_counts[1]}"
)
print(
    f"Test Target Balance     : 0 -> {test_counts[0]} | 1 -> {test_counts[1]}"
)




[TASK 2 SUCCESS] Data Partitioned & Scaled.

Training Features Shape : (233, 4)
Testing Features Shape  : (100, 4)
Train Target Balance    : 0 -> 102 | 1 -> 131
Test Target Balance     : 0 -> 44 | 1 -> 56


In [ ]:
# ==========================================================
# TASK 3 — LOGISTIC REGRESSION TRAINING
# ==========================================================

model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

print("\n[TASK 3 SUCCESS] Logistic Regression Model Trained.\n")
print("Model Intercept (Beta_0):", np.round(model.intercept_, 4))
print("\nFeature Coefficients (Beta_i):")

print(model.coef_[0])

for feature, coef in zip(X.columns, model.coef_[0]):
    print(f"  - {feature:<18}: {coef:7.4f}")




[TASK 3 SUCCESS] Logistic Regression Model Trained.

Model Intercept (Beta_0): [0.9766]

Feature Coefficients (Beta_i):
[ 3.7992416  -1.8719609   0.44451519 -0.39591728]
  - bill_length_mm    :  3.7992
  - bill_depth_mm     : -1.8720
  - flipper_length_mm :  0.4445
  - body_mass_g       : -0.3959


In [ ]:
# ==========================================================
# TASK 4 — SIGMOID PROBABILITIES
# ==========================================================

probabilities = model.predict_proba(X_test_scaled)[:, 1]

print("\n[TASK 4 SUCCESS] Probabilities Evaluated for Test Samples:\n")

for i in range(5):
    risk = "High Risk" if probabilities[i] > 0.50 else "Low Risk"

    print(
        f"Sample {i+1} | P(Anomalous): {probabilities[i]:.4f} | "
        f"Label: {y_test.iloc[i]} | Assigned: {risk}"
    )

# ==========================================================
# TASK 5 — DECISION BOUNDARY ANALYSIS
# ==========================================================

print("\n========== DECISION BOUNDARY SENSITIVITY ANALYSIS ==========\n")

threshold_results = {}

for threshold in [0.30, 0.50, 0.70]:

    predictions = (probabilities >= threshold).astype(int)

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)

    threshold_results[threshold] = (accuracy, precision, recall)

    print(
        f"Threshold: {threshold:.2f} | "
        f"Accuracy: {accuracy*100:.1f}% | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f}"
    )

print("\n============================================================")

# ==========================================================
# FINAL SUMMARY
# ==========================================================

base_accuracy = threshold_results[0.50][0]
max_prob = probabilities.max()
min_prob = probabilities.min()

coefs = dict(zip(X.columns, model.coef_[0]))

print("\n========== API-DRIVEN LOGISTIC REGRESSION & DECISION BOUNDARY ENGINE ==========\n")

print("Data Ingestion Status      : REST API Ingestion Successful (HTTP 200 OK)")
print(f"Master Dataset Records     : {len(df)} (Cleaned & Preprocessed)")
print(f"Features Included          : {X.shape[1]} Continuous Log Metrics ({', '.join(X.columns)})")
print("Target Output              : Is_Anomalous (Binary Classification: 0 = Standard, 1 = Anomalous)\n")

print("Model Training Metrics:")
print(f"- Stratified Train Split   : {len(y_train)} Records ({train_counts[0]} Normal / {train_counts[1]} Anomalous)")
print(f"- Stratified Test Split    : {len(y_test)} Records ({test_counts[0]} Normal / {test_counts[1]} Anomalous)")
print(f"- Base Test Accuracy       : {base_accuracy*100:.1f}% (at Default 0.50 Threshold)\n")

print("Sigmoid Probability Profiling:")
print(f"- Maximum Anomalous Prob   : {max_prob*100:.2f}%")
print(f"- Minimum Anomalous Prob   : {min_prob*100:.2f}%")
print(f"- Key Probability Drivers  : bill_length_mm ({coefs['bill_length_mm']:+.4f}), bill_depth_mm ({coefs['bill_depth_mm']:+.4f})\n")

print("Decision Threshold Tuning:")
for t, (acc, prec, rec) in threshold_results.items():
    print(f"- Threshold @ {t:.2f}: Accuracy {acc*100:.1f}% | Precision: {prec:.4f} | Recall: {rec:.4f}")

print("\nConclusion:")
print(
    "Dynamic REST API fetching ingests live CSV data over HTTP. "
    "Logistic Regression converts continuous features into calibrated "
    "sigmoid probabilities, allowing decision thresholds to be adjusted "
    "based on risk tolerance."
)


[TASK 4 SUCCESS] Probabilities Evaluated for Test Samples:

Sample 1 | P(Anomalous): 0.9962 | Label: 1 | Assigned: High Risk
Sample 2 | P(Anomalous): 0.0117 | Label: 0 | Assigned: Low Risk
Sample 3 | P(Anomalous): 0.9953 | Label: 1 | Assigned: High Risk
Sample 4 | P(Anomalous): 0.9975 | Label: 1 | Assigned: High Risk
Sample 5 | P(Anomalous): 0.9999 | Label: 1 | Assigned: High Risk

========== DECISION BOUNDARY SENSITIVITY ANALYSIS ==========

Threshold: 0.30 | Accuracy: 99.0% | Precision: 0.9825 | Recall: 1.0000
Threshold: 0.50 | Accuracy: 100.0% | Precision: 1.0000 | Recall: 1.0000
Threshold: 0.70 | Accuracy: 100.0% | Precision: 1.0000 | Recall: 1.0000


========== API-DRIVEN LOGISTIC REGRESSION & DECISION BOUNDARY ENGINE ==========

Data Ingestion Status      : REST API Ingestion Successful (HTTP 200 OK)
Master Dataset Records     : 333 (Cleaned & Preprocessed)
Features Included          : 4 Continuous Log Metrics (bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g)
Targe